In [ ]:
import cv2
import numpy as np

In [ ]:
# 탬플릿 추적
# 그냥 target이미지를 image에 갖다 대서 각 지점 마다 비슷한 점수 맵을 반환
image_gray, target_gray = None
result = cv2.matchTemplate(image_gray, target_gray, cv2.TM_CCOEFF_NORMED) # 점수맵 반환
min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(result)
# (이때 기준은 target의 좌상단, 즉 image 사이즈에 target 사이즈를 뺀 사이즈 점수 맵이 반환)

In [ ]:
# 배경 제거
# 활용도가 높진 않을 듯?
fgbg = cv2.createBackgroundSubtractorMOG2()
frame = None
fgbg.apply(frame)

In [ ]:
# optical flow
# 몇 개의 점만 추적
# 코드가 복잡함으로 사용하려면
# 4차시_03 코드 확인하길 바람
prevImg = None
nextImg = None
prevPt = None
termcriteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03) # |는 OR 비트 계산
cv2.goodFeaturesToTrack(prevImg, 200, 0.01, 10)
nextPt, status, err = cv2.calcOpticalFlowPyrLK(prevImg, nextImg, # 
                                     prevPt,None, criteria=termcriteria)

In [ ]:
# optical flow
# 모든 픽셀 추적
# 4차시_04 코드 확인 바람
# 4차시_07 코드도
prev, gray = None, None
cv2.calcOpticalFlowFarneback(prev, gray, None,
                                            0.5, # 이미지 피라미드 스케일(각 단계에서 이미지 50% 축소)
                                            3,   # 피라미드 레벨 수(3단계 피라미드 생성)
                                                 # 원본 >> 50% >> 25% (큰 움직임도 잡아냄)
                                            15,  # 윈도우 크기 (평균 이동을 위한 이웃 픽셀 수)
                                                 # 15*15 px 영역에서 계산
                                                 # 크면 부드러워지면서 세밀함 감소
                                                 # 작으면 세밀하지만 노이즈 증가
                                            3,   # 반복 횟수
                                            5,   # 다항식 확장 크기 (픽셀 이웃을 다항식으로 근사)
                                                 # 5*5 영역 (5는 빠른 처리용, 일반적 7)
                                            1.1, # 시그마 값 (가우시안 시그마- 가우시안 블러 >> 노이즈 제거)
                                            cv2.OPTFLOW_FARNEBACK_GAUSSIAN) # 플래그

In [ ]:
# 4차시_05 코드 확인
# mean shift
# 박스 형태로 객체 추적
# 박스 형태가 변환이 없기 때문에 추적에 어려움
# 물체의 hsv 중 h를 사용하여 추적

termination = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 1)
# 목표 정하기
# roi = frame[y:y + h, x:x + w]
roi_hsv = None
roi_hist = cv2.calcHist([roi_hsv], [0], None, [180], [0, 180])
# [roi_hsv] : HSV 형태로 변환한 입력 이미지 >> [] 리스트로 전달
# [0] : H 채널만 사용 (0번 채널)
# [180]: bin 개수 (히스토그램 구간 개수) 180개 bin
#  None  마스크 >> 전체 사용
# [0, 180]: H 채널 범위 (0도~180도)

# 4. 히스토그램 정규화 (최소 0, 최대 255)
cv2.normalize(roi_hist, roi_hist, 0, 255, cv2.NORM_MINMAX)


# 목표와 비슷한 객체 추적하기
# 전체 영상에 대해 ROI 히스토그램을 역투영(Back Projection) (***)
# dst는 각 픽셀이 ROI 색상과 얼마나 유사한지 나타내는 확률 맵이 됨. (0~255)
# [0]: H(색상) 채널 사용, [0, 180]: H 채널의 범위
# 1: scale
dst = cv2.calcBackProject([hsv], [0], roi_hist, [0, 180], 1) # 각 부분이 비슷한 정도를 수치로 나타냄

# 역투영 결과(dst)와 이전 추적 위치로 평균 이동(Mean Shift) 추적 실행
# ret: 반복 횟수, (x, y, w, h): 새로운 추적 위치
ret, (x, y, w, h) = cv2.meanShift(dst, (x, y, w, h), termination)



# cam shift
# 4차시_06 코드 확인
ret, track_window = cv2.CamShift(dst, (x, y, w, h), termination)